In [1]:
from CADETProcess.optimization import OptimizationProblem
optimization_problem = OptimizationProblem('batch_elution_single')

from examples.batch_elution.process import process
optimization_problem.add_evaluation_object(process)

optimization_problem.add_variable('cycle_time', lb=10, ub=600)
optimization_problem.add_variable('feed_duration.time', lb=10, ub=300)

optimization_problem.add_linear_constraint(
    ['feed_duration.time', 'cycle_time'], [1, -1]
)

[INFO 08-12 16:12:09] ax.storage.sqa_store.with_db_settings_base: Ax SQL storage initialized with SQLAlchemy 2.0.52


In [2]:
from CADETProcess.simulator import Cadet
process_simulator = Cadet()
process_simulator.evaluate_stationarity = True

optimization_problem.add_evaluator(process_simulator)

In [3]:
from CADETProcess.fractionation import FractionationOptimizer
frac_opt = FractionationOptimizer()

optimization_problem.add_evaluator(
    frac_opt,
    kwargs={
        'purity_required': [0.95, 0.95],
        'ignore_failed': False,
        'allow_empty_fractions': False,
    }
)

In [4]:
def callback(fractionation, individual, evaluation_object, callbacks_dir):
    fractionation.plot_fraction_signal(
        file_name=f'{callbacks_dir}/{individual.id}_{evaluation_object}_fractionation.png',
    )


optimization_problem.add_callback(
    callback, requires=[process_simulator, frac_opt]
)

In [5]:
from CADETProcess.performance import PerformanceProduct
ranking = [1, 1]
performance = PerformanceProduct(ranking=ranking)

optimization_problem.add_objective(
    performance, requires=[process_simulator, frac_opt], minimize=False,
)

In [6]:
from CADETProcess.optimization import U_NSGA3
optimizer = U_NSGA3()